In [11]:
import pandas as pd
import numpy as np
from fuzzywuzzy import process
import pycountry
from geopy.geocoders import Nominatim
import time

In [12]:
fla = pd.read_csv("fla_csvs/is_quantitative.csv")
unique_countries = fla['primary_region'].unique()
len(unique_countries)

4

In [13]:
geolocator = Nominatim(user_agent="geo_cleaner")
valid_countries = [country.name for country in pycountry.countries] 

In [14]:
#Define the function to clean country names
def clean_country(value):
    value  = str(value).strip() #to make sure the value is a string and removes any empty spaces around it 

    if pd.isna(value) or '@' in str(value) or str(value).isdigit() or len(str(value).strip()) < 2:
        print(f"Failed to process {value}, labelling as unknown.")
        return 'unknown'  #checks for bad or empty values, if the value is missing, has an @symbol, 
                          #is just a number, or is too short(less than 3 letters), treat it as unknown.
    
    #fuzzy match
    best_match, score = process.extractOne(str(value), valid_countries)
    if score >=80:
        return best_match #finds the closest matching real country from "valid_countries" list.
                          # if the score is 80 or higher (meaning, very sure), we use the matched name
    
    
    #Geopy fallback
    try:
        location = geolocator.geocode(value, language='en', addressdetails=True, timeout=10)
        if location and 'country' in location.raw['address']:
            return location.raw['address']['country']
    except:
        pass  # if fuzzy matching failed, we use geopy to guess the country by searching online.
              # if it finds a country, we return it. otherwise, we just skip using "pass"

    print(f"Failed to process {value}, labelling as unknown.")
    return 'unknown'  # we label it as unknown if both fuzzy matching and geopy fail
    

In [15]:
mapping = {}

# using a for loop to clean each value in the unique_values list
for val in unique_countries:
    cleaned = clean_country(val)
    mapping[val] = cleaned
    time.sleep(1) # respect geopy's rate limit (1 request per second)

#apply mapping to create a new country column (very fast method)
#df['clean_country'] = df['country'].map(mapping)

In [16]:
mapping

{'United Kingdom': 'United Kingdom',
 'Canada': 'Canada',
 'Germany': 'Germany',
 'UK': 'Ukraine'}

In [17]:
fla['clean_primary_region'] = fla['primary_region'].map(mapping)
clean_unique_countries = fla['clean_primary_region'].unique()
len(clean_unique_countries)

4

In [18]:
import pycountry_convert as pc

def country_to_continent(country_name):
    try:
        country_alpha2 = pc.country_name_to_country_alpha2(country_name)
        country_continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
        country_continent_name = pc.convert_continent_code_to_continent_name(country_continent_code)
        return country_continent_name
    except Exception as e:
        print(country_name)
        if country_name == 'Kosovo':
            return 'Europe'
        else:
            return "unknown"

In [19]:
fla['primary_continent'] = fla['clean_primary_region'].apply(country_to_continent)

In [20]:
fla.to_csv("fla_csvs/continent.csv")